In [0]:
#| default_exp write

## Writing and changing

Cell creation, targeted updates, documentation insertion, and examples.

In [ ]:
#| export
import tomllib
from pathlib import Path

from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import read_nb as _read_nb
from fastcore.nbio import write_nb as _write_nb
from fastcore.script import Param, call_parse
from nbdev.doclinks import nbdev_export

from nbskill.execute import _run_notebook_test
from nbskill.foundation import (
    _cell_hash, _cell_matches_hash, _cell_source, _clear_outputs, _cli_error,
    _cli_return, _find_cell_by_id, _find_cell_by_text, _load_cells_text,
    _one_chapter, _parse_cells, _parse_one_cell, _replace_cell, _tracked_call,
    _validate_code_cells,
)

In [ ]:
#| export
@call_parse
@_tracked_call
def write_nb(
    path: str,  # Notebook path
    cells: Param("Cell block text", str, opt=False, nargs="?") = "",  # Cells to write; use - to read stdin
    cells_file: str | None = None,  # Read cell block text from a UTF-8 file to avoid shell escaping
    before_id: str | None = None,  # Insert before this stable cell id
    after_id: str | None = None,  # Insert after this stable cell id
    chapter: str | None = None,  # Chapter title string or regex; missing chapters are created
    replace: bool = False,  # Replace the full notebook, or the selected chapter body
    cell_type: str = "code",  # Default type for cells without %% marker
    export: bool = True,  # Run nbdev-export after writing
    run_test: bool = False,  # Execute the notebook with execnb after writing
    run_style: bool = False,  # Run chstyle after writing
    style_strict: bool = False,  # Fail when chstyle finds hints
    validate_code: bool = True,  # Validate new Python code cells before writing
):
    "Write cells to a notebook using append, replace, id anchors, or chapters."
    if before_id and after_id: _cli_error("Use either before_id or after_id, not both")
    if (before_id or after_id) and chapter is not None: _cli_error("Use id-based insertion or chapter insertion, not both")
    if (before_id or after_id) and replace: _cli_error("Use id-based insertion or replace, not both")
    path = Path(path)
    cells = _load_cells_text(cells, cells_file)
    new_cells = _parse_cells(cells, cell_type)
    if validate_code: _validate_code_cells(new_cells)

    if replace and chapter is None:
        nb = new_nb(new_cells)
    else:
        nb = _read_nb(path) if path.exists() else new_nb([])
        if chapter is not None:
            span = _one_chapter(nb.cells, chapter, create=True)
            if replace:
                del nb.cells[span["start"] + 1:span["end"]]
                target = span["start"] + 1
            else:
                target = span["end"]
        elif before_id or after_id:
            idx, _ = _find_cell_by_id(nb.cells, before_id or after_id)
            target = idx if before_id else idx + 1
        else:
            target = len(nb.cells)
        for offset, cell in enumerate(new_cells):
            nb.cells.insert(target + offset, cell)

    _write_nb(nb, path)
    if export: nbdev_export(path=str(path))
    msg = f"Wrote {len(nb.cells)} cells to {path}"
    if replace: msg += " using replace"
    if chapter is not None: msg += f" in chapter {chapter!r}"
    if before_id: msg += f" before id={before_id}"
    if after_id: msg += f" after id={after_id}"
    if export: msg += " and exported with nbdev"
    print(msg)
    if run_test: _run_notebook_test(path)
    if run_style:
        print(f"Running chstyle on {path}")
        _run_chstyle(path, strict=style_strict)
    return _cli_return(path)

In [ ]:
import tempfile as _tempfile
from pathlib import Path as _Path
from fastcore.nbio import read_nb as _read_nb
from nbskill.write import update_cell, write_nb

with _tempfile.TemporaryDirectory() as td:
    path = _Path(td) / "write.ipynb"
    write_nb(str(path), "%%code\nvalue = 1\nvalue = value + 1", replace=True, export=False)
    cell = _read_nb(path).cells[0]
    update_cell(str(path), "value = 2", cell_id=cell.id, line_range="2", export=False)
    assert _read_nb(path).cells[0].source == "value = 1\nvalue = 2"
    update_cell(str(path), "", cell_id=cell.id, line_range="1", export=False)
    assert _read_nb(path).cells[0].source == "value = 2"

In [ ]:
#| export
def _save_nb(nb, path, export=True):
    _write_nb(nb, path)
    if export: nbdev_export(path=str(path))

In [ ]:
#| export
def _parse_line_range(line_range, n_lines):
    if line_range is None: return None
    value = str(line_range).strip()
    if not value: return None
    if ":" in value:
        start_s, end_s = value.split(":", 1)
        start = int(start_s) if start_s else 1
        end = int(end_s) if end_s else n_lines
    else:
        start = end = int(value)
    if start < 1 or end < start or end > n_lines:
        _cli_error(f"line_range must be 1-based and within 1:{n_lines}; got {line_range!r}")
    return start - 1, end


def _replace_line_range(source, line_range, new):
    lines = source.splitlines()
    start, end = _parse_line_range(line_range, len(lines) or 1)
    replacement = [] if new == "" else new.splitlines()
    return "\n".join([*lines[:start], *replacement, *lines[end:]])


@call_parse
@_tracked_call
def update_cell(
    path: str,  # Notebook path
    new: Param("Replacement cell source, replacement text, or line-range replacement", str, opt=False, nargs="?") = "",
    new_file: str | None = None,  # Read replacement text from a UTF-8 file
    cell_id: str | None = None,  # Stable notebook cell id to update
    old_str: str | None = None,  # Text to replace, or text used to find the target cell
    line_range: str | None = None,  # 1-based inclusive lines to replace, e.g. 3 or 3:5
    source_hash: str | None = None,  # Expected current source SHA256 prefix
    cell_type: str = "code",  # Default type for whole-cell replacements without %% marker
    export: bool = True,  # Run nbdev-export after writing
    run_test: bool = False,  # Execute the notebook with execnb after writing
    validate_code: bool = True,  # Validate changed Python code before writing
    dry_run: bool = False,  # Show the update plan without writing
):
    "Update one notebook cell by id, replace old_str, or replace a 1-based line range."
    if cell_id is None and old_str is None: _cli_error("Pass --cell_id, --old_str, or both")
    if line_range is not None and cell_id is None: _cli_error("Pass --cell_id with --line_range")
    path = Path(path)
    nb = _read_nb(path)
    new = _load_cells_text(new, new_file)

    idx, cell = _find_cell_by_id(nb.cells, cell_id) if cell_id else _find_cell_by_text(nb.cells, old_str)
    if old_str is not None and old_str not in _cell_source(cell):
        _cli_error(f"old_str was not found in id={cell.id}")
    if not _cell_matches_hash(cell, source_hash):
        actual = _cell_hash(cell, n=None)
        _cli_error(f"Hash mismatch for id={cell.id}: expected {source_hash}, actual {actual[:12]}")

    before_hash = _cell_hash(cell)
    if line_range is not None:
        replacement = _replace_line_range(_cell_source(cell), line_range, new)
        if validate_code and getattr(cell, "cell_type", None) == "code": _validate_code_cells([mk_cell(replacement)])
        after_hash = _cell_hash(replacement)
        mode = f"lines {line_range}"
        if not dry_run:
            cell.source = replacement
            _clear_outputs(cell)
    elif old_str is None:
        new_cell = _parse_one_cell(new, cell_type)
        if validate_code: _validate_code_cells([new_cell])
        _clear_outputs(new_cell)
        if not dry_run: _replace_cell(nb, idx, new_cell)
        after_hash = _cell_hash(new_cell)
        mode = "cell"
    else:
        replacement = _cell_source(cell).replace(old_str, new, 1)
        if validate_code and getattr(cell, "cell_type", None) == "code": _validate_code_cells([mk_cell(replacement)])
        after_hash = _cell_hash(replacement)
        mode = "text"
        if not dry_run:
            cell.source = replacement
            _clear_outputs(cell)

    msg = f"{'Dry run: would update' if dry_run else 'Updated'} {mode} id={cell.id} hash={before_hash}->{after_hash}"
    if dry_run:
        print(msg)
        return _cli_return(path)
    _write_nb(nb, path)
    if export: nbdev_export(path=str(path))
    if export: msg += " and exported with nbdev"
    print(msg)
    if run_test: _run_notebook_test(path)
    return _cli_return(path)

In [0]:
#| export
def _read_apply_nb_spec(spec_path):
    if spec_path == "-":
        import sys
        raw = sys.stdin.read()
        base = Path.cwd()
    else:
        path = Path(spec_path).expanduser()
        raw = path.read_text(encoding="utf-8")
        base = path.parent
    spec = tomllib.loads(raw)
    params = dict(spec.get("params", {}))
    for key, value in spec.items():
        if key != "params": params.setdefault(key, value)
    tool = params.pop("tool", params.pop("action", "write_nb"))
    cleanup = params.pop("cleanup", True)
    return tool, params, cleanup, base

In [0]:
#| export
def _resolve_sidecar_path(value, base):
    if value is None: return None
    path = Path(value).expanduser()
    return path if path.is_absolute() else base / path

In [0]:
#| export
def _remove_if_sidecar(path, base):
    path = Path(path).expanduser()
    try:
        resolved, root = path.resolve(), Path(base).resolve()
        resolved.relative_to(root)
    except (OSError, ValueError):
        return False
    if resolved.exists() and resolved.is_file():
        resolved.unlink()
        return True
    return False

In [0]:
#| export
def _cleanup_apply_nb_inputs(spec_path, params, base, cleanup):
    if not cleanup or spec_path == "-": return []
    removed = []
    for key in ("cells_file", "new_file"):
        if params.get(key) and _remove_if_sidecar(_resolve_sidecar_path(params[key], base), base):
            removed.append(str(_resolve_sidecar_path(params[key], base)))
    if _remove_if_sidecar(spec_path, base): removed.append(str(Path(spec_path).expanduser()))
    return removed

In [ ]:
#| export
@call_parse
@_tracked_call
def apply_nb(
    spec_path: Param("TOML operation file; use - to read TOML from stdin", str, opt=False, nargs="?") = "dev/nbskill-op.toml",
):
    "Apply a notebook operation from a TOML file and clean up dev sidecar files."
    tool, params, cleanup, base = _read_apply_nb_spec(spec_path)
    cleanup_params = dict(params)
    for key in ("cells_file", "new_file"):
        if params.get(key): params[key] = str(_resolve_sidecar_path(params[key], base))
    tools = {
        "write_nb": write_nb,
        "update_cell": update_cell,
    }
    if tool not in tools: _cli_error(f"Unknown apply_nb tool {tool!r}; expected one of {', '.join(tools)}")
    result = tools[tool](**params)
    removed = _cleanup_apply_nb_inputs(spec_path, cleanup_params, base, cleanup)
    if removed: print("Removed " + ", ".join(removed))
    return _cli_return(result)

In [0]:
import tempfile as _tempfile
from pathlib import Path as _Path

from fastcore.nbio import read_nb as _read_nb
from nbskill.write import apply_nb

with _tempfile.TemporaryDirectory() as td:
    root = _Path(td)
    dev = root / "dev"
    dev.mkdir()
    spec = dev / "nbskill-op.toml"
    spec.write_text(
        "\n".join([
            'tool = "write_nb"',
            'path = "demo.ipynb"',
            "replace = true",
            "export = false",
            'cells = """',
            "%%markdown",
            "## Scratch",
            "---",
            "%%code",
            "value = 3",
            '"""',
        ]),
        encoding="utf-8",
    )
    old_cwd = _Path.cwd()
    try:
        import os as _os
        _os.chdir(root)
        apply_nb(str(spec))
    finally:
        _os.chdir(old_cwd)
    assert not spec.exists()
    nb = _read_nb(root / "demo.ipynb")
    assert len(nb.cells) == 2
    assert nb.cells[1].source == "value = 3"